# 💸 월별 실제 인출액 입력

매달 포트폴리오에서 실제로 인출한 금액을 기록합니다.
- 시스템이 계산한 **권장 인출액**은 참고용
- **실제 인출액**을 직접 입력해야 의미 있는 데이터가 쌓입니다
- 국민연금 개시 후에는 생활비에서 연금을 차감한 금액을 입력하세요

In [1]:
import sqlite3, yaml
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
from datetime import date, datetime

PROJECT_ROOT = Path.cwd()
db_path      = PROJECT_ROOT / 'portfolio.db'

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
PENSION_CFG     = CONFIG.get('income', {}).get('national_pension', {})
base_pension    = PENSION_CFG.get('base_amount', 0)
pension_start   = PENSION_CFG.get('start_date', None)

# ── DB 마이그레이션: actual_amount 컬럼 추가 ─────────────────
conn = sqlite3.connect(db_path)
cur  = conn.cursor()
try:
    cur.execute('ALTER TABLE withdrawal_log ADD COLUMN actual_amount REAL DEFAULT NULL')
    conn.commit()
    print('✅ withdrawal_log 테이블에 actual_amount 컬럼 추가 완료')
except sqlite3.OperationalError:
    print('ℹ️  actual_amount 컬럼 이미 존재 — 정상')
conn.close()

display(HTML('<h2 style="color:#2c3e50;margin-bottom:4px">💸 실제 인출액 입력</h2><hr style="margin-top:0">'))

✅ withdrawal_log 테이블에 actual_amount 컬럼 추가 완료


In [3]:
# ── 이번 달 현황 표시 ────────────────────────────────────────
today      = date.today()
month_str  = today.strftime('%Y-%m')
month_disp = today.strftime('%Y년 %m월')

conn = sqlite3.connect(db_path)
cur  = conn.cursor()
cur.execute("""
    SELECT amount, actual_amount, note
    FROM withdrawal_log
    WHERE date LIKE ?
    ORDER BY date DESC LIMIT 1
""", (f'{month_str}%',))
row = cur.fetchone()
conn.close()

if row:
    recommended  = row[0] or 0
    existing_actual = row[1]  # None이면 미입력
    db_note      = row[2] or ''
else:
    recommended  = 0
    existing_actual = None
    db_note      = ''
    print('⚠️  이번 달 02_bucket_engine.ipynb를 먼저 실행하세요.')

# 국민연금 수령 여부
if pension_start:
    py, pm = map(int, pension_start.split('-'))
    if today >= date(py, pm, 1):
        inflation_rate = CONFIG.get('inflation', {}).get('assumed_rate', 0.025)
        years_since = (today.year - py) + (today.month - pm) / 12
        pension_now = base_pension * (1 + inflation_rate) ** years_since
    else:
        pension_now = 0
else:
    pension_now = 0

# 상단 정보 카드
status_color = '#27ae60' if existing_actual is not None else '#e67e22'
status_text  = f'입력 완료 ({existing_actual:,.0f}원)' if existing_actual is not None else '미입력'

html = f'''
<div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:16px;margin-bottom:16px;">
  <div style="font-size:16px;font-weight:bold;color:#2c3e50;margin-bottom:12px">{month_disp} 인출 현황</div>
  <table style="width:100%;font-size:14px;">
    <tr>
      <td style="color:#7f8c8d;width:180px">월 생활비</td>
      <td style="font-weight:bold">{MONTHLY_EXPENSE:,.0f}원</td>
    </tr>
    <tr>
      <td style="color:#7f8c8d">국민연금 수령액</td>
      <td style="font-weight:bold">{pension_now:,.0f}원 {'(수령 중)' if pension_now > 0 else f'(개시 전)'}</td>
    </tr>
    <tr>
      <td style="color:#7f8c8d">시스템 권장 인출액</td>
      <td style="font-weight:bold;color:#2980b9">{recommended:,.0f}원</td>
    </tr>
    <tr>
      <td style="color:#7f8c8d">실제 입력 상태</td>
      <td style="font-weight:bold;color:{status_color}">{status_text}</td>
    </tr>
  </table>
</div>
'''
display(HTML(html))

월 생활비,"5,000,000원"
국민연금 수령액,0원 (개시 전)
시스템 권장 인출액,"5,000,000원"
실제 입력 상태,미입력


In [5]:
# ── 입력 위젯 ────────────────────────────────────────────────
style  = {'description_width': '160px'}
layout = widgets.Layout(width='420px')

# 연도·월 선택
w_year = widgets.IntText(
    value=today.year,
    description='연도',
    style=style, layout=widgets.Layout(width='260px')
)
w_month = widgets.Dropdown(
    options=list(range(1, 13)),
    value=today.month,
    description='월',
    style=style, layout=widgets.Layout(width='260px')
)

# 실제 인출액
default_val = int(existing_actual) if existing_actual is not None else int(recommended) if recommended else int(MONTHLY_EXPENSE - pension_now)
w_actual = widgets.IntText(
    value=default_val,
    description='실제 인출액 (원)',
    style=style, layout=layout
)

w_memo = widgets.Text(
    value='',
    description='메모 (선택)',
    placeholder='예: 의료비 추가 지출 등',
    style=style, layout=layout
)

display(HTML('<h3 style="color:#34495e">✏️ 인출액 입력</h3>'))
display(widgets.HBox([w_year, w_month]))
display(w_actual, w_memo)

# 참고 계산
ref_label = widgets.HTML()

def update_ref(*args):
    val  = w_actual.value
    diff = val - (MONTHLY_EXPENSE - pension_now)
    color = '#e74c3c' if diff > 0 else '#27ae60'
    sign  = '+' if diff >= 0 else ''
    ref_label.value = (
        f'<div style="margin-top:4px;font-size:13px;color:#7f8c8d">'
        f'생활비 - 연금 = {MONTHLY_EXPENSE - pension_now:,.0f}원 기준 '
        f'<span style="color:{color};font-weight:bold">{sign}{diff:,.0f}원</span></div>'
    )

w_actual.observe(update_ref, names='value')
update_ref()
display(ref_label)

IntText(value=5000000, description='실제 인출액 (원)', layout=Layout(width='420px'), style=DescriptionStyle(descript…

Text(value='', description='메모 (선택)', layout=Layout(width='420px'), placeholder='예: 의료비 추가 지출 등', style=TextSt…

HTML(value='<div style="margin-top:4px;font-size:13px;color:#7f8c8d">생활비 - 연금 = 5,000,000원 기준 <span style="col…

In [8]:
# ── 저장 버튼 ────────────────────────────────────────────────
btn_save = widgets.Button(
    description='💾 저장',
    button_style='success',
    layout=widgets.Layout(width='150px', height='40px')
)
out = widgets.Output()

def on_save(b):
    with out:
        clear_output()
        target_month = f'{w_year.value:04d}-{w_month.value:02d}'
        actual_val   = w_actual.value
        memo         = w_memo.value.strip()

        conn = sqlite3.connect(db_path)
        cur  = conn.cursor()

        # 해당 월 레코드 확인
        cur.execute(
            'SELECT id, amount FROM withdrawal_log WHERE date LIKE ?',
            (f'{target_month}%',)
        )
        existing = cur.fetchone()

        if existing:
            # actual_amount 업데이트
            extra_note = f' | 메모: {memo}' if memo else ''
            cur.execute(
                'UPDATE withdrawal_log SET actual_amount=? WHERE date LIKE ?',
                (actual_val, f'{target_month}%')
            )
            conn.commit()
            conn.close()
            recommended_val = existing[1] or 0
            diff = actual_val - recommended_val
            diff_str = f'+{diff:,.0f}원' if diff >= 0 else f'{diff:,.0f}원'
            display(HTML(
                f'<div style="color:#27ae60;font-size:14px;font-weight:bold">'
                f'✅ {target_month} 실제 인출액 저장 완료!</div>'
                f'<div style="font-size:13px;color:#7f8c8d;margin-top:4px">'
                f'권장 {recommended_val:,.0f}원 → 실제 {actual_val:,.0f}원 ({diff_str})'
                f'</div>'
            ))
        else:
            # 해당 월 레코드 자체가 없음 — 02_bucket_engine 먼저 실행 필요
            conn.close()
            display(HTML(
                f'<div style="color:#e74c3c;font-size:14px">'
                f'❌ {target_month} 레코드가 없습니다.<br>'
                f'02_bucket_engine.ipynb를 먼저 실행한 후 입력해 주세요.</div>'
            ))

btn_save.on_click(on_save)
display(HTML('<br>'))
display(btn_save)
display(out)

Button(button_style='success', description='💾 저장', layout=Layout(height='40px', width='150px'), style=ButtonSt…

Output()

In [9]:
# ── 인출 이력 (권장 vs 실제 비교) ────────────────────────────
display(HTML('<h3 style="color:#34495e;margin-top:24px">📊 인출 이력 — 권장 vs 실제</h3>'))

conn = sqlite3.connect(db_path)
hist = pd.read_sql_query("""
    SELECT
        substr(date,1,7)    AS 월,
        amount              AS 권장_인출액,
        actual_amount       AS 실제_인출액,
        note                AS 비고
    FROM withdrawal_log
    ORDER BY date DESC
    LIMIT 24
""", conn)
conn.close()

if hist.empty:
    print('인출 이력이 없습니다. 02_bucket_engine.ipynb를 먼저 실행하세요.')
else:
    hist['권장_인출액'] = hist['권장_인출액'].apply(
        lambda x: f'{x:,.0f}원' if pd.notna(x) else '-'
    )
    hist['실제_인출액'] = hist['실제_인출액'].apply(
        lambda x: f'{x:,.0f}원' if pd.notna(x) else '⬅️ 미입력'
    )
    display(HTML(hist.to_html(index=False, border=0,
        classes='table', justify='left').replace(
        '<table', '<table style="font-size:13px;width:100%"'
    )))

월,권장_인출액,실제_인출액,비고
2026-05,"5,000,000원","5,000,000원","생활비 5,000,000원 | 연금 개시까지 59개월"
